# NB05 Public-key signatures — Chapter 5

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB05-1: toy elliptic curve y^2 = x^3 + 7 (mod 97) — point addition, doubling, scalar multiplication (for study only, never for real use)
p, a, b = 97, 0, 7          # same shape as secp256k1 (a=0, b=7), only smaller
O = None                     # point at infinity (identity element)

def on_curve(P):
    if P is O: return True
    x, y = P
    return (y * y - (x ** 3 + a * x + b)) % p == 0

def inv(z): return pow(z, p - 2, p)              # multiplicative inverse (Fermat's little theorem)

def add(P, Q):
    if P is O: return Q
    if Q is O: return P
    (x1, y1), (x2, y2) = P, Q
    if x1 == x2 and (y1 + y2) % p == 0: return O   # P + (-P) = O
    if P == Q: lam = (3 * x1 * x1 + a) * inv(2 * y1) % p
    else:      lam = (y2 - y1) * inv(x2 - x1) % p
    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def mul(k, P):                                    # double-and-add
    R = O
    while k:
        if k & 1: R = add(R, P)
        P = add(P, P)
        k >>= 1
    return R

points = [(x, y) for x in range(p) for y in range(p) if on_curve((x, y))]
print("number of points on the curve (including the point at infinity) n =", len(points) + 1)
G = min(points)                                   # take the point with the smallest x as the base point
print("base point G =", G)
print("2G =", add(G, G), " 3G =", mul(3, G), " 79G =", mul(79, G), "(None means the point at infinity)")
d = 23                                            # toy private key: never use for real
Q = mul(d, G)
print("private key d = %d → public key Q = [d]G = %s" % (d, Q))
found = next(k for k in range(1, 79) if mul(k, G) == Q)
print("d recovered from Q by exhaustive search: d =", found, "(instant with only 79 points; the real curve is on the scale of 2^256)")

number of points on the curve (including the point at infinity) n = 79
base point G = (1, 28)
2G = (68, 81)  3G = (53, 38)  79G = None (None means the point at infinity)
private key d = 23 → public key Q = [d]G = (61, 28)
d recovered from Q by exhaustive search: d = 23 (instant with only 79 points; the real curve is on the scale of 2^256)


In [ ]:
# NB05-2: toy ECDSA — signing and verification, failure when one bit changes, (r, n-s) also valid (for study only, never for real use)
import hashlib
n = 79                                            # order of the base point G

def inv_n(z): return pow(z, n - 2, n)

def msg_int(M: bytes) -> int:
    """Message hash as an integer. Since n is small, reduce the SHA-256 result modulo n (a simplification for illustration)."""
    return int.from_bytes(hashlib.sha256(M).digest(), "big") % n

def sign(d, M, k):
    e = msg_int(M)
    R = mul(k, G)
    r = R[0] % n
    s = inv_n(k) * (e + r * d) % n
    assert r != 0 and s != 0
    return (r, s)

def verify(Q, M, sig):
    r, s = sig
    if not (1 <= r <= n - 1 and 1 <= s <= n - 1): return False
    e = msg_int(M)
    u = e * inv_n(s) % n
    v = r * inv_n(s) % n
    R1 = add(mul(u, G), mul(v, Q))
    return R1 is not O and R1[0] % n == r

M = b"Pay 1 coin to Bob"
k = 17                                            # one-time secret number: fixed here for illustration (in practice, a secure random number)
sig = sign(d, M, k)
print("message:", M, "| signature (r, s) =", sig)
print("normal verification      :", verify(Q, M, sig))
print("one character changed    :", verify(Q, b"Pay 2 coin to Bob", sig))
print("someone else's public key:", verify(mul(24, G), M, sig))
r, s = sig
print("(r, n-s) =", (r, (n - s) % n), "is also valid:", verify(Q, M, (r, (n - s) % n)), " ← ECDSA malleability")

message: b'Pay 1 coin to Bob' | signature (r, s) = (17, 45)
normal verification      : True
one character changed    : False
someone else's public key: False
(r, n-s) = (17, 34) is also valid: True  ← ECDSA malleability


In [ ]:
# NB05-3: use the secret number k twice, and the private key can be computed from two public signatures (for study only)
M1, M2 = b"Pay 1 coin to Bob", b"Pay 5 coins to Carol"
k_reused = 31
(r1, s1), (r2, s2) = sign(d, M1, k_reused), sign(d, M2, k_reused)
e1, e2 = msg_int(M1), msg_int(M2)
print("the two signatures have the same r:", r1 == r2, "(r =", r1, ")")
k_rec = (e1 - e2) * inv_n(s1 - s2) % n
d_rec = (s1 * k_rec - e1) * inv_n(r1) % n
print("recovered k =", k_rec, "| recovered private key d =", d_rec, "| actual d =", d)

the two signatures have the same r: True (r = 52 )
recovered k = 31 | recovered private key d = 23 | actual d = 23


In [ ]:
# NB05-4: the real secp256k1 parameters (SEC 2 §2.4.1), checked with the standard library alone (for study only, never for real use)
import hashlib
p = 2**256 - 2**32 - 2**9 - 2**8 - 2**7 - 2**6 - 2**4 - 1
a, b = 0, 7
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
G = (Gx, Gy)
print("p in base 16:", hex(p).upper()[2:])
print("G is on the curve            :", on_curve(G))
print("[n]G is the point at infinity:", mul(n, G) is O)
# recover y from the compressed form 02||x: since p ≡ 3 (mod 4), y = (x^3+7)^((p+1)/4). The prefix 02 means y is even.
y_cand = pow((Gx**3 + 7) % p, (p + 1) // 4, p)
y_even = y_cand if y_cand % 2 == 0 else p - y_cand
print("y recovered from the compressed form == y of the uncompressed form:", y_even == Gy)

d_toy = 1                                         # study private key 1 → the public key is G itself. This is not real key generation.
Qx, Qy = mul(d_toy, G)
compressed = bytes([2 + (Qy & 1)]) + Qx.to_bytes(32, "big")
print("compressed public key for d=1:", compressed.hex())
h160 = hashlib.new("ripemd160", hashlib.sha256(compressed).digest()).digest()
print("RIPEMD160(SHA256(K)), 20 bytes:", h160.hex())
print("The first byte of an uncompressed 65-byte public key is 0x04; the public key in the genesis output of Chapter 12 has this form.")

p in base 16: FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
G is on the curve            : True
[n]G is the point at infinity: True
y recovered from the compressed form == y of the uncompressed form: True
compressed public key for d=1: 0279be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798
RIPEMD160(SHA256(K)), 20 bytes: 751e76e8199196d454941c45d1b3a323f1433bd6
The first byte of an uncompressed 65-byte public key is 0x04; the public key in the genesis output of Chapter 12 has this form.
